notebook_presentacion_ridge.ipynb

Generado para Google Colab.

# Análisis del desempeño académico SIMCE mediante IDPS y variables contextuales
## Presentación 2 — Modelo final (Ridge Regression)

Este notebook estudia **cómo se asocian los Indicadores de Desarrollo Personal y
Social (IDPS), junto con variables contextuales del establecimiento, con los
resultados SIMCE de Matemática y Lectura, según el nivel educativo evaluado.**

El modelo principal es **Ridge Regression** (`RidgeCV`), una regresión lineal
regularizada. Se elige Ridge —y no una regresión lineal ordinaria (OLS)— como
respuesta directa al feedback de la Presentación 1 sobre **multicolinealidad**:
Ridge penaliza (encoge) los coeficientes inestables que aparecen cuando los
predictores están correlacionados, entregando coeficientes más robustos sin perder
interpretabilidad.

## Evolución metodológica respecto a la Presentación 1

En la Presentación 1 se usó una regresión lineal múltiple sobre un dataset
provisorio. A partir del feedback del profesor se hicieron ajustes importantes:

- **Objetivo descriptivo-explicativo, no predictivo.** El profesor observó que
  presentar el trabajo como "predicción" no calza cuando las variables IDPS y el
  SIMCE pertenecen al mismo período. Asumimos explícitamente un marco
  **contemporáneo**: estudiamos *asociaciones* del mismo año, sin afirmar predicción
  futura ni causalidad. (Por eso se descartó la línea predictiva con desfase T-1: el
  objetivo del proyecto es comprender la relación, no anticipar resultados futuros.)
- **NSE como variable ordinal** (`nse_ord`, 1 a 5), en vez de tratarlo como categórica.
- **Análisis de simetría (skewness)** de puntajes e IDPS.
- **Diagnóstico de multicolinealidad (VIF)** entre los IDPS, que motiva el uso de Ridge.
- **`tipo_prueba`** como variable explicativa que une Matemática y Lectura en un solo
  modelo (lo sugirió el profesor para no construir modelos completamente independientes).
- **Validación más exigente:** comparación explícita train vs test, validación cruzada
  agrupada por establecimiento, y justificación de la métrica (MAE).

## Pregunta de investigación

¿Qué relación existe entre los IDPS, las variables contextuales de los establecimientos
y los puntajes SIMCE de Matemática y Lectura, y cómo cambia esa relación entre cursos?

## Objetivo general

Comprender cómo se asocian los indicadores de desarrollo personal y social y el
contexto del establecimiento con el rendimiento SIMCE, diferenciando por curso y
asignatura, para identificar qué factores se relacionan con mejores o peores resultados.

## Enfoque OSEMN

1. **Obtain:** carga del dataset final por curso.
2. **Scrub:** nulos, categorías, NSE ordinal, skewness, outliers por curso y VIF.
3. **Explore:** análisis descriptivo y correlaciones.
4. **Model:** Ridge Regression por curso y asignatura (8 modelos) + variante sin NSE.
5. **iNterpret:** comparación de métricas, coeficientes e interpretación educativa.

> Enfoque **contemporáneo y descriptivo**: estudia asociaciones del mismo período. No
> afirma causalidad ni predicción futura.

In [ ]:
# 0) Imports y configuración
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display
except ImportError:                      # permite correr como script fuera de Colab
    display = print

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

DATA_DIR = Path("/content")              # en Colab los CSV van a /content
CURSOS = ["4b", "6b", "8b", "2m"]
CURSO_NOMBRE = {"4b": "4° básico", "6b": "6° básico", "8b": "8° básico", "2m": "II medio"}
IDPS_NOMBRE = {"ind_am": "Autoestima", "ind_cc": "Convivencia",
               "ind_hv": "Hábitos", "ind_pf": "Participación"}
ASIGNATURAS = [("prom_mate2m_rbd", "Matemática"), ("prom_lect2m_rbd", "Lectura")]
PALETA = {"Matemática": "steelblue", "Lectura": "coral"}
FEATURES_IDPS = ["ind_am", "ind_cc", "ind_hv", "ind_pf"]
TARGETS = ["prom_mate2m_rbd", "prom_lect2m_rbd"]

# Rejilla de alphas que RidgeCV explora en cada modelo (penalización L2).
ALPHAS_RIDGE = np.logspace(-3, 4, 50)

# 1) OBTAIN — Carga de datos finales

Se usan los cuatro datasets por curso consolidados desde el pipeline del proyecto
(`dataset_historico_{4b,6b,8b,2m}.csv`). Cada archivo contiene puntajes SIMCE, IDPS y
variables contextuales por establecimiento (`rbd`), año (`agno`) y curso (`curso`).

In [ ]:
# 1.1) Carga
def cargar_datos(data_dir=DATA_DIR):
    rutas = {c: data_dir / f"dataset_historico_{c}.csv" for c in CURSOS}
    faltan = [str(p) for p in rutas.values() if not p.exists()]
    if faltan:                            # fallback: buscar bajo el cwd y /content
        for raiz in [Path.cwd(), Path("/content")]:
            hit = next(raiz.rglob("dataset_historico_4b.csv"), None) if raiz.exists() else None
            if hit:
                return cargar_datos(hit.parent)
        raise FileNotFoundError(f"No encontré los CSV por curso. Faltan: {faltan}")
    partes = []
    for c in CURSOS:
        t = pd.read_csv(rutas[c])
        print(f"  dataset_historico_{c}.csv: {t.shape[0]:,} filas × {t.shape[1]} columnas")
        partes.append(t)
    return pd.concat(partes, ignore_index=True)


df_raw = cargar_datos()
print(f"\nDataset unido: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")
display(df_raw.head())

# 1.2) Panorama inicial (registros, años, cobertura)
print("Años disponibles:", sorted(df_raw["agno"].dropna().unique()))
print("\nRegistros por curso:")
display(df_raw["curso"].astype(str).str.strip().value_counts()
        .reindex(CURSOS).rename(index=CURSO_NOMBRE).to_frame("n_registros"))
print("Registros por año y curso:")
display(pd.crosstab(df_raw["agno"], df_raw["curso"].astype(str).str.strip()).reindex(columns=CURSOS))

# 2) SCRUB — Limpieza y preparación

El dataset ya viene consolidado. Aquí se estandarizan categorías (NSE, dependencia,
ruralidad), se revisan nulos y rangos, y se tratan outliers por curso. También se
codifica el NSE como variable **ordinal**, respondiendo al feedback de la Presentación 1.

## Nota: codificación ordinal del NSE

`cod_grupo` representa el nivel socioeconómico del establecimiento y tiene un orden
natural: `Bajo < Medio bajo < Medio < Medio alto < Alto`. Por eso se crea `nse_ord`
(1 a 5) respetando ese orden. Es la misma idea que `OrdinalEncoder`, pero explícita y
fácil de defender. En el modelo `nse_ord` se estandariza junto con los IDPS, así su
coeficiente es comparable en magnitud con ellos (cambio en SIMCE por una desviación
estándar de NSE).

In [ ]:
# 2.1) Normalización de categorías y NSE ordinal
NSE_MAP = {**{k: "Bajo" for k in ["1", "1.0", "Bajo"]},
           **{k: "Medio bajo" for k in ["2", "2.0", "Medio bajo"]},
           **{k: "Medio" for k in ["3", "3.0", "Medio"]},
           **{k: "Medio alto" for k in ["4", "4.0", "Medio alto"]},
           **{k: "Alto" for k in ["5", "5.0", "Alto"]}}
NSE_ORD = {"Bajo": 1, "Medio bajo": 2, "Medio": 3, "Medio alto": 4, "Alto": 5}
COLS_REQ = ["rbd", "agno", "curso", *TARGETS, "cod_grupo", "nse_ord",
            "cod_rural_rbd", "cod_depe2", *FEATURES_IDPS]


def preparar(df):
    df = df.copy()
    for c in ["cod_grupo", "cod_depe2", "cod_rural_rbd", "curso"]:
        df[c] = df[c].astype(str).str.strip()
    df["cod_grupo"] = df["cod_grupo"].replace(NSE_MAP)
    df["nse_ord"] = df["cod_grupo"].map(NSE_ORD)
    for c in ["rbd", "agno", *TARGETS, *FEATURES_IDPS, "nse_ord"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["curso_nombre"] = df["curso"].map(CURSO_NOMBRE)
    return df


df_prep = preparar(df_raw)
print("Categorías detectadas:")
for col in ["cod_grupo", "cod_rural_rbd", "cod_depe2", "curso"]:
    print(f"- {col}: {sorted(df_prep[col].dropna().unique())}")

print("\nNulos por columna relevante:")
display(df_prep[COLS_REQ].isna().sum().to_frame("nulos"))

df_scrub = df_prep.dropna(subset=COLS_REQ).copy()
print(f"Filas: {len(df_prep):,} → tras dropna: {len(df_scrub):,} "
      f"({len(df_prep) - len(df_scrub):,} eliminadas)")

# 2.2) Revisión de rangos por curso
print("Rangos (min / mean / max) de puntajes e IDPS por curso:")
display(df_scrub.groupby("curso")[TARGETS + FEATURES_IDPS]
        .agg(["min", "mean", "max"]).reindex(CURSOS).rename(index=CURSO_NOMBRE).round(2))

## Nota: análisis de simetría (skewness)

El feedback de la Presentación 1 pidió revisar la simetría de las variables. Regla de
lectura: `|s|<0.5` aprox. simétrica · `0.5≤|s|<1` asimetría moderada · `|s|≥1` alta.
No cambia el modelo automáticamente, pero justifica si las variables son razonables
para una regresión lineal.

In [ ]:
# 2.3) Skewness por curso
skew_tbl = (df_scrub.groupby("curso")[TARGETS + FEATURES_IDPS].skew()
            .reindex(CURSOS).rename(index=CURSO_NOMBRE).round(3))
print("Skewness por curso (puntajes e IDPS):")
display(skew_tbl)

## Nota: outliers por IQR calculado dentro de cada curso

El rango intercuartílico (IQR) se calcula **por curso** porque mezclar 4° básico, 6°,
8° e II medio puede marcar como outlier un valor que solo pertenece a otra escala de
comparación educativa. Se filtran los puntajes SIMCE fuera de `[Q1−1.5·IQR, Q3+1.5·IQR]`.

In [ ]:
# 2.4) Filtro de outliers IQR por curso
def quitar_outliers_iqr(df):
    keep = pd.Series(True, index=df.index)
    for _, g in df.groupby("curso"):
        kc = pd.Series(True, index=g.index)
        for col in TARGETS:
            q1, q3 = g[col].quantile(0.25), g[col].quantile(0.75)
            iqr = q3 - q1
            kc &= g[col].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)
        keep.loc[g.index] = kc
    return df.loc[keep].copy()


df_modelo = quitar_outliers_iqr(df_scrub)
print(f"Filas: {len(df_scrub):,} → tras IQR: {len(df_modelo):,} "
      f"({len(df_scrub) - len(df_modelo):,} eliminadas)")
display(df_modelo.groupby("curso_nombre").size().reindex(
    [CURSO_NOMBRE[c] for c in CURSOS]).to_frame("registros_finales"))

## Multicolinealidad entre IDPS (VIF) — por qué Ridge

El VIF mide cuánto se "explica" cada IDPS a partir de los otros. `VIF<5` indica
multicolinealidad leve; `5–10` moderada; `>10` alta. Aun con VIF bajo, los predictores
están algo correlacionados, y eso hace que en OLS los coeficientes individuales puedan
volverse inestables. **Ridge ataca exactamente ese problema:** penaliza los coeficientes
grandes/inestables, reduciendo su varianza. Por eso es el modelo principal de esta entrega.

In [ ]:
# 2.5) VIF por curso
def vif(df_num):
    filas = []
    for col in df_num.columns:
        X = df_num.drop(columns=col)
        r2 = LinearRegression().fit(X, df_num[col]).score(X, df_num[col])
        filas.append({"variable": IDPS_NOMBRE.get(col, col),
                      "VIF": np.inf if 1 - r2 < 1e-12 else 1 / (1 - r2)})
    return pd.DataFrame(filas)


vif_tbl = pd.concat([vif(df_modelo[df_modelo["curso"] == c][FEATURES_IDPS]).assign(
    curso=CURSO_NOMBRE[c]) for c in CURSOS], ignore_index=True)
print("VIF de los IDPS por curso:")
display(vif_tbl.pivot(index="variable", columns="curso", values="VIF")
        [[CURSO_NOMBRE[c] for c in CURSOS]].round(2))

# 3) EXPLORE — Análisis descriptivo

Se compara la distribución de SIMCE entre cursos, la correlación global entre SIMCE,
IDPS y NSE, y la correlación IDPS↔SIMCE dentro de cada curso. Esto verifica si tiene
sentido modelar cada nivel por separado.

In [ ]:
# 3.1) Distribución de SIMCE por curso y matriz de correlación global
plot_df = df_modelo.melt(id_vars=["curso_nombre"], value_vars=TARGETS,
                         var_name="_c", value_name="puntaje")
plot_df["asignatura"] = plot_df["_c"].map(dict(ASIGNATURAS))
plot_df["curso_nombre"] = pd.Categorical(
    plot_df["curso_nombre"], [CURSO_NOMBRE[c] for c in CURSOS], ordered=True)

plt.figure(figsize=(10, 4.5))
sns.boxplot(data=plot_df, x="curso_nombre", y="puntaje", hue="asignatura", palette=PALETA)
plt.title("Distribución de puntajes SIMCE por curso y asignatura")
plt.xlabel("Curso"); plt.ylabel("Puntaje SIMCE"); plt.tight_layout(); plt.show()

corr_labels = {"prom_mate2m_rbd": "Matemática", "prom_lect2m_rbd": "Lectura",
               "ind_am": "Autoestima", "ind_cc": "Convivencia", "ind_hv": "Hábitos",
               "ind_pf": "Participación", "nse_ord": "NSE"}
plt.figure(figsize=(8, 6))
sns.heatmap(df_modelo[list(corr_labels)].corr().rename(index=corr_labels, columns=corr_labels),
            annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlación global: SIMCE, IDPS y NSE"); plt.tight_layout(); plt.show()

# 3.2) Correlación entre IDPS (apoya el uso de Ridge)
plt.figure(figsize=(6.5, 5))
sns.heatmap(df_modelo[FEATURES_IDPS].corr().rename(index=IDPS_NOMBRE, columns=IDPS_NOMBRE),
            annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlación entre indicadores IDPS")
plt.xticks(rotation=30, ha="right"); plt.yticks(rotation=0); plt.tight_layout(); plt.show()

# 3.3) Correlación IDPS ↔ SIMCE por curso (tablas)
filas_corr = []
for curso in CURSOS:
    g = df_modelo[df_modelo["curso"] == curso]
    for idps in FEATURES_IDPS:
        for target, asig in ASIGNATURAS:
            filas_corr.append({"indicador": IDPS_NOMBRE[idps], "asignatura": asig,
                               "curso": CURSO_NOMBRE[curso], "corr": g[idps].corr(g[target])})
corr_idps_simce = pd.DataFrame(filas_corr)
for asig in ["Matemática", "Lectura"]:
    print(f"Correlación IDPS ↔ SIMCE — {asig}")
    display(corr_idps_simce[corr_idps_simce["asignatura"] == asig]
            .pivot(index="indicador", columns="curso", values="corr")
            [[CURSO_NOMBRE[c] for c in CURSOS]].round(3))

# 4) MODEL — Ridge Regression por curso y asignatura

Se entrenan **8 modelos** (4 cursos × 2 asignaturas: Matemática y Lectura) usando
`RidgeCV`. Este enfoque permite comparar directamente los coeficientes de IDPS entre
niveles y asignaturas.

## Variables del modelo
**Targets:** `prom_mate2m_rbd` (Matemática), `prom_lect2m_rbd` (Lectura).
**Predictores:** IDPS (`ind_am`, `ind_cc`, `ind_hv`, `ind_pf`) + NSE ordinal
estandarizado (`nse_ord`) + ruralidad (`cod_rural_rbd`) + dependencia (`cod_depe2`).
Los IDPS y `nse_ord` se estandarizan para que sus coeficientes sean comparables;
ruralidad y dependencia van con one-hot (coeficientes = diferencia vs categoría base).

## Por qué Ridge (`RidgeCV`)
- **Penalización L2:** encoge los coeficientes hacia cero, estabilizándolos cuando los
  IDPS están correlacionados (lo que vimos en VIF). No elimina variables.
- **`alpha` automático por modelo:** `RidgeCV` elige el nivel de penalización por
  validación cruzada interna, distinto para cada curso/asignatura.
- **Misma interpretación que OLS:** los coeficientes siguen siendo asociaciones
  ajustadas (manteniendo el resto constante), solo que más conservadoras.

## Decisiones de validación
- **Split agrupado por `rbd`:** un mismo establecimiento no aparece a la vez en train y test.
- **Validación cruzada agrupada (GroupKFold):** evalúa estabilidad respetando la unidad establecimiento.
- **NSE ordinal estandarizado:** permite comparar su magnitud con los IDPS.
- **One-hot con categoría base (`drop="first"`):** evita redundancia entre categorías.

In [ ]:
# 4.1) Funciones de modelado (reutilizables: Ridge, OLS y variante sin NSE)
def metricas(yr, yp):
    return {"MAE": mean_absolute_error(yr, yp),
            "RMSE": np.sqrt(mean_squared_error(yr, yp)), "R2": r2_score(yr, yp)}


def hacer_ridge():
    return RidgeCV(alphas=ALPHAS_RIDGE, cv=5)


def hacer_ols():
    return LinearRegression()


def pipeline(feat_num, feat_cat, regresor):
    pre = ColumnTransformer([("num", StandardScaler(), feat_num),
                             ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), feat_cat)])
    return Pipeline([("pre", pre), ("reg", regresor)])


def correr_por_curso(feat_num, feat_cat, hacer_regresor):
    """Devuelve (métricas, coeficientes) para los 8 modelos curso×asignatura."""
    met, coef = [], []
    for curso in CURSOS:
        d = df_modelo[df_modelo["curso"] == curso]
        for target, asig in ASIGNATURAS:
            data = d.dropna(subset=feat_num + feat_cat + [target, "rbd"])
            X, y, g = data[feat_num + feat_cat], data[target], data["rbd"]
            tr, te = next(GroupShuffleSplit(1, test_size=0.2, random_state=42).split(X, y, g))
            m = pipeline(feat_num, feat_cat, hacer_regresor()).fit(X.iloc[tr], y.iloc[tr])
            mtr = metricas(y.iloc[tr], m.predict(X.iloc[tr]))
            mte = metricas(y.iloc[te], m.predict(X.iloc[te]))
            r2cv = cross_validate(pipeline(feat_num, feat_cat, hacer_regresor()), X, y, groups=g,
                                  cv=GroupKFold(5), scoring="r2")["test_score"].mean()
            alpha = getattr(m.named_steps["reg"], "alpha_", np.nan)
            met.append({"curso": CURSO_NOMBRE[curso], "asignatura": asig, "n": len(data),
                        "alpha": alpha, "MAE_train": mtr["MAE"], "MAE_test": mte["MAE"],
                        "RMSE_test": mte["RMSE"], "R2_train": mtr["R2"],
                        "R2_test": mte["R2"], "R2_CV": r2cv})
            nombres = (m.named_steps["pre"].named_transformers_["num"].get_feature_names_out(feat_num).tolist()
                       + m.named_steps["pre"].named_transformers_["cat"].get_feature_names_out(feat_cat).tolist())
            for v, c in zip(nombres, m.named_steps["reg"].coef_):
                coef.append({"curso": CURSO_NOMBRE[curso], "asignatura": asig, "variable": v, "coef": c})
    return pd.DataFrame(met), pd.DataFrame(coef)


FEAT_NUM = ["ind_am", "ind_cc", "ind_hv", "ind_pf", "nse_ord"]
FEAT_CAT = ["cod_rural_rbd", "cod_depe2"]
met_principal, coef_principal = correr_por_curso(FEAT_NUM, FEAT_CAT, hacer_ridge)
orden = [CURSO_NOMBRE[c] for c in CURSOS]
met_principal["curso"] = pd.Categorical(met_principal["curso"], orden, ordered=True)
met_principal = met_principal.sort_values(["curso", "asignatura"]).reset_index(drop=True)

print("Métricas de los 8 modelos Ridge (alpha + train / test / validación cruzada):")
display(met_principal.round(3))

# 4.2) Desempeño por curso (R² y MAE) y coeficientes IDPS
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, metr, titulo in zip(axes, ["R2_test", "MAE_test"], ["R² test", "MAE test"]):
    sns.barplot(data=met_principal, x="curso", y=metr, hue="asignatura", palette=PALETA, ax=ax)
    ax.set_title(titulo); ax.set_xlabel("Curso"); ax.tick_params(axis="x", rotation=15)
plt.suptitle("Modelo Ridge: desempeño por curso y asignatura"); plt.tight_layout(); plt.show()

coef_idps = coef_principal[coef_principal["variable"].isin(FEATURES_IDPS)].copy()
coef_idps["indicador"] = coef_idps["variable"].map(IDPS_NOMBRE)
coef_idps["curso"] = pd.Categorical(coef_idps["curso"], orden, ordered=True)
for asig in ["Matemática", "Lectura"]:
    print(f"Coeficientes IDPS estandarizados (Ridge) — {asig}")
    display(coef_idps[coef_idps["asignatura"] == asig]
            .pivot(index="indicador", columns="curso", values="coef")
            [[CURSO_NOMBRE[c] for c in CURSOS]].round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, asig in zip(axes, ["Matemática", "Lectura"]):
    sns.lineplot(data=coef_idps[coef_idps["asignatura"] == asig], x="curso", y="coef",
                 hue="indicador", marker="o", ax=ax)
    ax.axhline(0, color="black", lw=0.8); ax.set_title(f"Coeficientes IDPS — {asig}")
    ax.set_xlabel("Curso"); ax.set_ylabel("Coef. estandarizado"); ax.tick_params(axis="x", rotation=15)
plt.suptitle("Asociación IDPS→SIMCE por curso (Ridge)"); plt.tight_layout(); plt.show()

# 4.3) Importancia de variables por modelo (todas las variables)
fig, axes = plt.subplots(4, 2, figsize=(14, 18))
for i, curso in enumerate(CURSOS):
    for j, (target, asig) in enumerate(ASIGNATURAS):
        ax = axes[i, j]
        c = coef_principal[(coef_principal["curso"] == CURSO_NOMBRE[curso])
                           & (coef_principal["asignatura"] == asig)].copy()
        c = c.reindex(c["coef"].abs().sort_values(ascending=False).index)
        colores = [PALETA[asig] if v > 0 else "gray" for v in c["coef"]]
        ax.barh(c["variable"], c["coef"], color=colores)
        ax.axvline(0, color="black", lw=0.8); ax.invert_yaxis()
        ax.set_title(f"{asig} · {CURSO_NOMBRE[curso]}")
plt.suptitle("Importancia de variables (Ridge)", y=1.001)
plt.tight_layout(); plt.show()

## 4.4) Robustez: Ridge vs OLS

Como chequeo, se entrenan los mismos 8 modelos con regresión lineal ordinaria (OLS) y
se comparan. **Si Ridge y OLS entregan resultados casi idénticos, es buena señal:**
significa que la multicolinealidad detectada en VIF era leve y no estaba desestabilizando
gravemente los coeficientes. Ridge actúa entonces como una "red de seguridad" que
confirma la estabilidad del modelo, sin sacrificar ajuste.

In [ ]:
# 4.4) Comparación Ridge vs OLS
met_ols, _ = correr_por_curso(FEAT_NUM, FEAT_CAT, hacer_ols)
met_ols["curso"] = pd.Categorical(met_ols["curso"], orden, ordered=True)
met_ols = met_ols.sort_values(["curso", "asignatura"]).reset_index(drop=True)
comp_ro = met_principal[["curso", "asignatura", "R2_test", "MAE_test"]].merge(
    met_ols[["curso", "asignatura", "R2_test", "MAE_test"]],
    on=["curso", "asignatura"], suffixes=("_Ridge", "_OLS"))
comp_ro["delta_R2"] = comp_ro["R2_test_Ridge"] - comp_ro["R2_test_OLS"]
print("Ridge vs OLS por modelo (test):")
display(comp_ro.round(4))
print(f"R² test medio — Ridge: {met_principal['R2_test'].mean():.4f} | "
      f"OLS: {met_ols['R2_test'].mean():.4f} | Δ medio: {comp_ro['delta_R2'].mean():+.4f}")

plt.figure(figsize=(9, 4.5))
comp_long = comp_ro.melt(id_vars=["curso", "asignatura"],
                         value_vars=["R2_test_Ridge", "R2_test_OLS"],
                         var_name="modelo", value_name="R2_test")
comp_long["etiqueta"] = comp_long["curso"].astype(str) + " · " + comp_long["asignatura"].str[:4]
comp_long["modelo"] = comp_long["modelo"].str.replace("R2_test_", "")
sns.barplot(data=comp_long, x="etiqueta", y="R2_test", hue="modelo")
plt.title("R² test: Ridge vs OLS (prácticamente idénticos)")
plt.xlabel(""); plt.ylabel("R² test"); plt.xticks(rotation=30, ha="right")
plt.tight_layout(); plt.show()

## 4.5) Variante sin NSE — ¿cuánto del "efecto IDPS" es en realidad NSE?

En Chile el nivel socioeconómico domina parte importante de las diferencias
educativas, por lo que puede ocultar la lectura de los IDPS. Se reentrenan los 8
modelos Ridge quitando solo `nse_ord`. Si el R² baja, el NSE aportaba información real;
si los coeficientes IDPS suben, parte de su asociación estaba mezclada con el contexto
socioeconómico.

In [ ]:
# 4.5) Comparación con / sin NSE (Ridge)
met_sin_nse, coef_sin_nse = correr_por_curso(FEATURES_IDPS, FEAT_CAT, hacer_ridge)
comp = met_principal[["curso", "asignatura", "R2_test", "MAE_test"]].merge(
    met_sin_nse[["curso", "asignatura", "R2_test", "MAE_test"]],
    on=["curso", "asignatura"], suffixes=("_con_NSE", "_sin_NSE"))
comp["delta_R2"] = comp["R2_test_sin_NSE"] - comp["R2_test_con_NSE"]
comp["delta_MAE"] = comp["MAE_test_sin_NSE"] - comp["MAE_test_con_NSE"]
print("Impacto de quitar NSE (Ridge, test):")
display(comp.round(3))

plt.figure(figsize=(9, 4.5))
sns.barplot(data=comp, x="curso", y="delta_R2", hue="asignatura", palette=PALETA)
plt.axhline(0, color="black", lw=0.8)
plt.title("Caída de R² al quitar NSE (más negativo = NSE aportaba más)")
plt.xlabel("Curso"); plt.ylabel("Δ R² (sin − con)"); plt.tight_layout(); plt.show()

# 5) Modelos complementarios (Ridge)

Dos modelos de apoyo, ambos con `RidgeCV`, que responden observaciones puntuales del
profesor sin reemplazar al modelo principal por curso:

- **Global:** todos los cursos juntos (`curso` como control de escala) → panorama y línea base.
- **Unificado `tipo_prueba`:** Matemática y Lectura apiladas en una sola variable objetivo;
  el coeficiente de `tipo_prueba` mide la brecha sistemática entre asignaturas (lo pidió el profe).

In [ ]:
# 5) Global + unificado (Ridge)
print("MODELO GLOBAL Ridge (todos los cursos):")
for target, asig in ASIGNATURAS:
    fn, fc = FEAT_NUM, ["cod_rural_rbd", "cod_depe2", "curso"]
    data = df_modelo.dropna(subset=fn + fc + [target, "rbd"])
    X, y, g = data[fn + fc], data[target], data["rbd"]
    tr, te = next(GroupShuffleSplit(1, test_size=0.2, random_state=42).split(X, y, g))
    m = pipeline(fn, fc, hacer_ridge()).fit(X.iloc[tr], y.iloc[tr])
    mte = metricas(y.iloc[te], m.predict(X.iloc[te]))
    print(f"  {asig:11s} R²_test={mte['R2']:.3f}  MAE_test={mte['MAE']:.2f}  alpha={m.named_steps['reg'].alpha_:.3f}")

du = df_modelo.melt(id_vars=["rbd", "curso", "nse_ord", "cod_rural_rbd", "cod_depe2", *FEATURES_IDPS],
                    value_vars=TARGETS, var_name="_c", value_name="puntaje")
du["tipo_prueba"] = du["_c"].map(dict(ASIGNATURAS))
du = du.dropna(subset=["puntaje"])
fn, fc = FEAT_NUM, ["cod_rural_rbd", "cod_depe2", "curso", "tipo_prueba"]
X, y, g = du[fn + fc], du["puntaje"], du["rbd"]
tr, te = next(GroupShuffleSplit(1, test_size=0.2, random_state=42).split(X, y, g))
m = pipeline(fn, fc, hacer_ridge()).fit(X.iloc[tr], y.iloc[tr])
mte = metricas(y.iloc[te], m.predict(X.iloc[te]))
nombres = (m.named_steps["pre"].named_transformers_["num"].get_feature_names_out(fn).tolist()
           + m.named_steps["pre"].named_transformers_["cat"].get_feature_names_out(fc).tolist())
coef_tp = {v: c for v, c in zip(nombres, m.named_steps["reg"].coef_) if v.startswith("tipo_prueba")}
print(f"\nMODELO UNIFICADO Ridge (tipo_prueba): R²_test={mte['R2']:.3f}  MAE_test={mte['MAE']:.2f}")
print(f"  coef {list(coef_tp)[0]} = {list(coef_tp.values())[0]:.2f} "
      f"(Matemática vs Lectura, a igualdad del resto)")

# 6) iNTERPRET — Síntesis e interpretación

Se conectan los resultados con la pregunta del proyecto y con la rúbrica.

In [ ]:
# 6.1) Síntesis cuantitativa
print("Mejor → peor R² test (Ridge, modelo principal):")
display(met_principal.sort_values("R2_test", ascending=False)
        [["curso", "asignatura", "alpha", "MAE_test", "RMSE_test", "R2_test"]].round(3).reset_index(drop=True))

br = met_principal[["curso", "asignatura", "MAE_train", "MAE_test", "R2_train", "R2_test", "R2_CV"]].copy()
br["brecha_R2_train_test"] = (br["R2_train"] - br["R2_test"]).round(3)
print("Brecha train–test y comparación con validación cruzada (estabilidad):")
display(br.round(3))

## Conclusiones (números reales de esta corrida)

**1. El NSE es el predictor dominante y su peso crece con el nivel.** Estandarizado,
pasa de ~10 pts en 4° básico a ~27 pts en II medio (Matemática): 2–4× el IDPS más influyente.

**2. Quitar el NSE empeora el ajuste, y el deterioro escala con el curso:** Δ R² ≈ −0,08
en básica hasta **−0,19 en II medio**. Buena parte del "efecto IDPS" es, en realidad, NSE.

**3. Entre los IDPS, Convivencia (`ind_cc`) es el más estable y dominante** (coef ≈ +5 a +9).
Participación (`ind_pf`) es inestable / cambia de signo → no interpretar como efecto.

**4. El ajuste mejora con el nivel educativo:** R² test ~0,37–0,43 en básica vs **0,61
(mate) / 0,56 (lect) en II medio**. MAE entre 13 y 19 puntos SIMCE.

**5. Sin sobreajuste:** en los 8 modelos R²_test ≈ R²_CV y la brecha train–test es ínfima (~0,001).

**6. Ridge ≈ OLS (Δ R² ≈ 0,000):** la regularización casi no mueve las métricas → confirma
que la multicolinealidad (VIF<5) era leve y que los coeficientes son estables. Ridge se
adopta por robustez metodológica, no por un salto en el ajuste.

**7. `tipo_prueba` ≈ −1,1:** la brecha mate–lectura es pequeña → comparten estructura,
lo que justifica que el modelo unificado funcione casi tan bien como los separados.

**Métrica elegida — MAE (principal):** error en puntos SIMCE reales, interpretable para
directivos ("erramos ±X puntos"). R²: bondad de ajuste comparable entre cursos. RMSE:
vigila errores grandes. Se reporta MAE como métrica de negocio y R²/RMSE como apoyo.

## ¿Qué enfoque responde nuestro objetivo? (la elección)

El **modelo Ridge por curso y asignatura (8 modelos)** es el que **responde directamente
nuestra pregunta**: permite ver *cómo cambia* la asociación IDPS↔SIMCE entre 4° básico e
II medio y entre Matemática y Lectura. Es el **modelo principal** de la presentación.

Los otros dos son **complementarios** y atienden observaciones puntuales del profesor,
sin reemplazarlo:
- **Global** → da el panorama y una línea base comparativa.
- **Unificado `tipo_prueba`** → responde la sugerencia de no construir modelos
  totalmente independientes, y cuantifica la brecha entre asignaturas.

## Cómo responde esta entrega al feedback de la Presentación 1

| Observación del profesor | Cómo se aborda |
|---|---|
| Objetivo es estrategia, no fin (5,7) | Objetivo reformulado a **comprender asociaciones** (descriptivo-explicativo). |
| "Predictivo" pero datos contemporáneos (5,5) | Se asume **explícitamente** el marco contemporáneo; se descarta la línea predictiva. |
| Explorar `tipo_prueba` (6,0) | Modelo unificado con `tipo_prueba` como variable explicativa. |
| Skewness, multicolinealidad, NSE ordinal (5,7) | Skewness por curso, **VIF**, **NSE ordinal**, y **Ridge** para la multicolinealidad. |
| Comparar train vs test, justificar MAE (5,2) | Tabla **train vs test + CV** en los 8 modelos; **MAE** justificada como métrica principal. |

## Limitaciones
- Los coeficientes representan **asociaciones, no causalidad**.
- El análisis es **contemporáneo** (IDPS y SIMCE del mismo año); no anticipa resultados futuros.
- Los IDPS presentan multicolinealidad leve → interpretar coeficientes con cautela
  (Ridge la mitiga, no la elimina).
- `RidgeCV` elige `alpha` con validación cruzada interna no agrupada por `rbd`: una
  simplificación frente a una búsqueda totalmente agrupada por establecimiento.
- No todos los cursos tienen los mismos años de SIMCE evaluados.